# pgvectorをベクトルデータベースとして使用し、レコードをベクトル検索するサンプル

## パッケージをインポート

In [ ]:
import psycopg2
from sentence_transformers import SentenceTransformer
import os
import numpy as np

## 埋め込みモデル初期化

In [ ]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
model

## 検索キーワードを定義し、埋め込みモデルでベクトル化する

In [ ]:
SEARCH_KEYWORD = '日本の都市'

In [ ]:
search_keyword_embedding = model.encode([SEARCH_KEYWORD])[0]
print(f'Search keyword: {SEARCH_KEYWORD}')
print(f'Embedding shape: {search_keyword_embedding.shape}')
search_keyword_embedding

## PostgreSQLへ接続

In [ ]:
db_config = {
    'user': os.environ['PGVECTOR_POSTGRES_USER'],
    'password': os.environ['PGVECTOR_POSTGRES_PASSWORD'],
    'host': 'llm-rag-examples-pgvector',
    'database': os.environ['PGVECTOR_POSTGRES_DB'],
    'port': '5432'
}

db_config

In [ ]:
conn = psycopg2.connect(**db_config)
cursor = conn.cursor()

## 方法1: pgvectorの近接検索機能を使用

In [ ]:
cursor.execute("""
    SELECT id, text, embedding <-> %s::vector AS distance 
    FROM text_embeddings 
    ORDER BY embedding <-> %s::vector 
    LIMIT 2
""", (search_keyword_embedding.tolist(), search_keyword_embedding.tolist()))

results = cursor.fetchall()
print(f"検索結果 (pgvector距離ベース):")
print(f"検索キーワード: {SEARCH_KEYWORD}")
print()

for i, (id, text, distance) in enumerate(results, 1):
    print(f'{i}. ID: {id}, 距離: {distance:.4f}, テキスト: {text}')

## 方法2: 全レコードを取得してPythonで類似度計算

In [ ]:
cursor.execute('SELECT id, text, embedding FROM text_embeddings')

## 検索結果の各レコードのベクトルと検索キーワードのベクトル積を計算する

In [ ]:
search_result: list = []
for (id, text, embedding) in cursor:
    print(f'ID: {id}, Text: {text}')
    print(f'Embedding type: {type(embedding)}')
    print(f'Embedding sample: {str(embedding)[:100]}...')
    
    # pgvectorから取得したベクトルデータをnumpy配列に変換
    if isinstance(embedding, str):
        # 文字列の場合、"[1.0, 2.0, 3.0]" 形式から数値配列に変換
        embedding_str = embedding.strip('[]')
        embedding_list = [float(x.strip()) for x in embedding_str.split(',')]
        embedding_array = np.array(embedding_list)
    else:
        # 既にリストや配列の場合
        embedding_array = np.array(embedding)
    
    dot_product = np.dot(embedding_array, search_keyword_embedding)
    search_result.append({
        'id': id,
        'text': text,
        'embedding': embedding_array,
        'dot_product': dot_product,
    })
    print(f'Dot product: {dot_product:.4f}')
    print()

In [ ]:
cursor.close()
conn.close()

In [ ]:
search_result

## ベクトル積の大きい順番にソートして表示する

In [ ]:
sorted_result_full = sorted(search_result, key=lambda x: x['dot_product'], reverse=True)

In [ ]:
sorted_result = [(l.get('id'), l.get('text'), l.get('dot_product')) for l in sorted_result_full]

In [ ]:
print(f"検索結果 (ベクトル積ベース):")
print(f"検索キーワード: {SEARCH_KEYWORD}")
print()

for i, (id, text, dot_product) in enumerate(sorted_result, 1):
    print(f'{i}. ID: {id}, ベクトル積: {dot_product:.4f}, テキスト: {text}')

## 比較: pgvector距離 vs ベクトル積

- **pgvector距離**: 小さいほど類似
- **ベクトル積**: 大きいほど類似

どちらも同じ順序で結果が返されることを確認できます。